# Train Site-Specific 9.6 km Momentum Emulators

Train terrain-specific residual U-Net models for fixed 9.6 km boxes and evaluate whether `mass solver + ML residual` closely emulates WindNinja momentum output on same-terrain held-out samples.

Default run queue:

- `breck_tenmile_9p6_specific_lcp_canopy_v1`
- `keystone_9p6_specific_lcp_canopy_v1`

This notebook treats HRRR-only same-terrain test performance as the main operational metric and controlled-only performance as a stress test. It also writes a scorecard with season, direction-sector, high-wind, canopy, and lee/windward breakdowns.


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else Path.cwd() / 'ml/residual_unet/outputs/colab_local'
LOCAL_ROOT = Path('/content/windninja_ml') if IN_COLAB else Path.cwd()
LOCAL_DATA_ROOT = Path('/content/data') if IN_COLAB else Path.cwd() / 'ml/residual_unet/data/processed'
ARTIFACT_DIR = LOCAL_ROOT / 'artifacts'
REPO_DIR = LOCAL_ROOT / 'repo'
RESULT_ROOT = DRIVE_ROOT / 'results'

GCP_PROJECT = os.environ.get('GCP_PROJECT', 'spring-nova-475120-r0')
GCS_BUCKET = os.environ.get('GCS_BUCKET', 'mwn-ml-general-9p6-spring-nova-475120-r0')
USE_GCS_ARTIFACTS = True
SYNC_RESULTS_TO_GCS = True
FORCE_DOWNLOAD_CODE = True
FORCE_DOWNLOAD_DATA = False
FORCE_UNPACK_CODE = True
FORCE_UNPACK_DATA = False
RESUME_IF_AVAILABLE = True

RUN_KEYS = ['breck', 'keystone']
RUN_CONFIGS = {
    'breck': {
        'dataset_name': 'breck_tenmile_9p6_specific_lcp_canopy_v1',
        'run_name': 'breck_tenmile_9p6_specific_lcp_canopy_v1',
        'config_name': 'breck_tenmile_9p6_specific_lcp_canopy_v1.yaml',
    },
    'keystone': {
        'dataset_name': 'keystone_9p6_specific_lcp_canopy_v1',
        'run_name': 'keystone_9p6_specific_lcp_canopy_v1',
        'config_name': 'keystone_9p6_specific_lcp_canopy_v1.yaml',
    },
}

TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 2
PREFETCH_FACTOR = 4
PROGRESS_EVERY = 100
WORST_CASE_LIMIT = 20

print('IN_COLAB', IN_COLAB)
print('RUN_KEYS', RUN_KEYS)
print('RESULT_ROOT', RESULT_ROOT)
print('ARTIFACT_DIR', ARTIFACT_DIR)


## Authenticate, Download, And Unpack

This cell mounts Drive, pulls the packaged code and terrain-specific dataset ZIPs from GCS, and unpacks them onto local Colab disk for faster training. Results still write to Drive so they survive runtime shutdown.


In [ ]:
if IN_COLAB:
    from google.colab import auth, drive
    auth.authenticate_user()
    drive.mount('/content/drive')

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

if USE_GCS_ARTIFACTS and IN_COLAB:
    subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)


def gcs_cp(name: str, destination: Path, *, force: bool = False) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not force:
        print('Found', destination)
        return
    source = f'gs://{GCS_BUCKET}/drive_upload/{name}'
    print('Copying', source, '->', destination)
    subprocess.run(['gcloud', 'storage', 'cp', source, str(destination)], check=True)


def unpack_zip(zip_path: Path, destination: Path, *, force: bool = False, required_path: Path | None = None) -> None:
    if required_path is not None and required_path.exists() and not force:
        print('Already unpacked:', required_path)
        return
    if destination.exists() and force:
        print('Removing stale unpacked directory:', destination)
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    print('Unpacking', zip_path, '->', destination)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)

code_zip = ARTIFACT_DIR / 'residual_unet_code.zip'
gcs_cp('residual_unet_code.zip', code_zip, force=FORCE_DOWNLOAD_CODE)
unpack_zip(code_zip, REPO_DIR, force=FORCE_UNPACK_CODE, required_path=REPO_DIR / 'ml/residual_unet/train.py')

for key in RUN_KEYS:
    info = RUN_CONFIGS[key]
    dataset_zip = ARTIFACT_DIR / f"{info['dataset_name']}_dataset.zip"
    gcs_cp(dataset_zip.name, dataset_zip, force=FORCE_DOWNLOAD_DATA)
    unpack_zip(
        dataset_zip,
        LOCAL_DATA_ROOT,
        force=FORCE_UNPACK_DATA,
        required_path=LOCAL_DATA_ROOT / info['dataset_name'] / 'manifest.csv',
    )

print('Repo:', REPO_DIR)
print('Data root:', LOCAL_DATA_ROOT)
print('Result root:', RESULT_ROOT)
print('Local disk free GB:', round(shutil.disk_usage('/content' if IN_COLAB else '.').free / 1024**3, 1))


## Install And Check Runtime


In [ ]:
requirements = REPO_DIR / 'ml/residual_unet/requirements.txt'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
else:
    print('Skipping install outside Colab')

import torch
print('torch', torch.__version__)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('gpu_memory_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## Inspect Datasets And Splits

Each dataset should contain HRRR paired samples plus the controlled 15-degree and 7.5-degree midpoint stress-test cases for the same terrain box.


In [ ]:
sys.path.insert(0, str(REPO_DIR))
from ml.residual_unet.config import load_config
from ml.residual_unet.dataset import filter_rows

RUN_STATE = {}
for key in RUN_KEYS:
    info = RUN_CONFIGS[key]
    data_dir = LOCAL_DATA_ROOT / info['dataset_name']
    config_path = REPO_DIR / 'ml/residual_unet/configs' / info['config_name']
    summary = json.loads((data_dir / 'dataset_summary.json').read_text())
    rows = list(csv.DictReader((data_dir / 'manifest.csv').open()))
    config = load_config(config_path)
    train_rows = filter_rows(rows, 'train')
    val_rows = filter_rows(rows, 'val')
    test_rows = filter_rows(rows, 'test')
    print('\n' + '=' * 80)
    print('run:', info['run_name'])
    print('dataset:', data_dir)
    print('config:', config_path)
    print('sample_count:', summary.get('sample_count'))
    print('input_channels:', summary.get('input_channels'))
    print('split_counts:', summary.get('split_counts'))
    print('planned train/val/test:', len(train_rows), len(val_rows), len(test_rows))
    print('by source:')
    for source, count in Counter(row['source_dataset'] for row in rows).most_common():
        print(f'  {source}: {count}')
    RUN_STATE[key] = {
        'info': info,
        'data_dir': data_dir,
        'config_path': config_path,
        'summary': summary,
        'rows': rows,
    }


## Train Terrain-Specific Models

This directly calls the training function so progress prints every epoch and every `PROGRESS_EVERY` batches. Leave `RESUME_IF_AVAILABLE=True` to continue cleanly after an interrupted runtime.


In [ ]:
import importlib

if 'ml.residual_unet.train' in sys.modules:
    importlib.reload(sys.modules['ml.residual_unet.train'])
from ml.residual_unet.config import apply_overrides, load_config
import ml.residual_unet.train as train_module


def result_paths(run_name: str) -> dict:
    result_dir = RESULT_ROOT / run_name
    return {
        'result_dir': result_dir,
        'checkpoint_dir': result_dir / 'checkpoints',
        'log_csv': result_dir / 'train_log.csv',
        'eval_root': result_dir / 'eval',
        'scorecard_dir': result_dir / 'scorecard',
    }

for key in RUN_KEYS:
    state = RUN_STATE[key]
    info = state['info']
    paths = result_paths(info['run_name'])
    paths['checkpoint_dir'].mkdir(parents=True, exist_ok=True)
    paths['log_csv'].parent.mkdir(parents=True, exist_ok=True)
    resume = paths['checkpoint_dir'] / 'latest.pt'
    resume = resume if RESUME_IF_AVAILABLE and resume.exists() else None
    overrides = {
        'data.processed_dir': str(state['data_dir']),
        'data.batch_size': TRAIN_BATCH_SIZE,
        'data.num_workers': NUM_WORKERS,
        'data.prefetch_factor': PREFETCH_FACTOR,
        'data.pin_memory': True,
        'training.checkpoint_dir': str(paths['checkpoint_dir']),
        'training.log_csv': str(paths['log_csv']),
        'training.progress_every': PROGRESS_EVERY,
    }
    config = apply_overrides(load_config(state['config_path']), overrides)
    print('\n' + '=' * 80)
    print('direct train() call')
    print('run_name:', info['run_name'])
    print('config:', state['config_path'])
    print('checkpoint_dir:', paths['checkpoint_dir'])
    print('log_csv:', paths['log_csv'])
    print('resume:', resume)
    print('overrides:', json.dumps(overrides, indent=2))
    train_module.train(config, resume=resume)


## Evaluate Sources And Build Emulator Scorecards

This writes standard source-specific metrics plus a terrain-specific scorecard. The scorecard is the main validation artifact for the emulator goal: it checks same-terrain held-out HRRR, controlled stress cases, high wind pixels, direction sectors, seasons, canopy bins, and lee/windward slope bins.


In [ ]:
if 'ml.residual_unet.evaluate' in sys.modules:
    importlib.reload(sys.modules['ml.residual_unet.evaluate'])
if 'ml.residual_unet.emulator_scorecard' in sys.modules:
    importlib.reload(sys.modules['ml.residual_unet.emulator_scorecard'])
import ml.residual_unet.evaluate as evaluate_module
from ml.residual_unet.emulator_scorecard import write_emulator_scorecard

for key in RUN_KEYS:
    state = RUN_STATE[key]
    info = state['info']
    paths = result_paths(info['run_name'])
    checkpoint = paths['checkpoint_dir'] / 'best.pt'
    assert checkpoint.exists(), checkpoint
    source_names = list(state['summary']['source_datasets'])
    print('\n' + '=' * 80)
    print('evaluating run:', info['run_name'])
    print('checkpoint:', checkpoint)
    for source in source_names:
        out = paths['eval_root'] / source
        print('evaluating source:', source)
        metrics = evaluate_module.evaluate(
            checkpoint,
            state['data_dir'],
            out,
            split='test',
            batch_size=EVAL_BATCH_SIZE,
            source_datasets=[source],
            num_workers=NUM_WORKERS,
            pin_memory=True,
            prefetch_factor=PREFETCH_FACTOR,
            max_figures=5,
        )
        print(source, json.dumps(metrics, indent=2))

    print('writing scorecard:', paths['scorecard_dir'])
    scorecard = write_emulator_scorecard(
        checkpoint,
        state['data_dir'],
        paths['scorecard_dir'],
        split='test',
        batch_size=EVAL_BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        prefetch_factor=PREFETCH_FACTOR,
        worst_case_limit=WORST_CASE_LIMIT,
    )
    overall = next(row for row in scorecard['metric_rows'] if row['group_type'] == 'overall')
    print('scorecard overall:', json.dumps(overall, indent=2))


## Compare All Result Folders

This keeps the progression report current for every run under `MyDrive/windninja_ml/results`, including older general models and the new terrain-specific models.


In [ ]:
if 'ml.residual_unet.compare_results' in sys.modules:
    importlib.reload(sys.modules['ml.residual_unet.compare_results'])
from ml.residual_unet.compare_results import compare_results

COMPARISON_DIR = RESULT_ROOT / '_comparison'
comparison = compare_results(RESULT_ROOT, COMPARISON_DIR)
print('comparison rows:', len(comparison['rows']))
print('comparison report:', COMPARISON_DIR / 'comparison_report.md')
print('run summaries:')
for row in comparison['run_summaries']:
    print(
        row['run_name'],
        'sources=', row['source_count'],
        'ml_rmse=', round(row['ml_vector_rmse'], 3),
        'improvement=', round(row['vector_rmse_improvement_percent'], 1),
    )


## Package Results Back To GCS

Drive already has the checkpoints, logs, eval metrics, figures, scorecards, and comparison report. This final cell syncs them to GCS for local download and backup.


In [ ]:
print('Result files:')
for key in RUN_KEYS:
    run_name = RUN_CONFIGS[key]['run_name']
    result_dir = RESULT_ROOT / run_name
    print('\n' + run_name)
    for path in sorted(result_dir.rglob('*')):
        if path.is_file():
            print(' ', path.relative_to(result_dir))

if SYNC_RESULTS_TO_GCS and IN_COLAB:
    subprocess.run(['gcloud', 'config', 'set', 'project', GCP_PROJECT], check=True)
    for key in RUN_KEYS:
        run_name = RUN_CONFIGS[key]['run_name']
        result_dir = RESULT_ROOT / run_name
        gcs_dir = f'gs://{GCS_BUCKET}/colab_results/{run_name}'
        print('Syncing', result_dir, '->', gcs_dir)
        subprocess.run(['gcloud', 'storage', 'rsync', '-r', str(result_dir), gcs_dir], check=True)
    comparison_gcs_dir = f'gs://{GCS_BUCKET}/colab_results/_comparison'
    subprocess.run(['gcloud', 'storage', 'rsync', '-r', str(COMPARISON_DIR), comparison_gcs_dir], check=True)
    subprocess.run(['gcloud', 'storage', 'ls', f'gs://{GCS_BUCKET}/colab_results/'], check=True)
else:
    print('Skipping GCS sync')
